# Stage 3: Benchmark evaluation and metric validation

**Document Unwarping | 21 November 2026**

This notebook evaluates saved Stage 2 predictions. It checks the official UVDoc line-metric implementation first, then runs the full 50-image benchmark and prepares optional visual-metric inputs for DocUNet and DIR300.

Each output has its own directory under `runs/evaluation/`. Existing datasets, model weights, and Stage 2 runs remain unchanged. MATLAB-dependent published scores are only calculated when the original evaluator is available.

## 01. Mount the existing repository

Use the same project root as Stage 1 and Stage 2. Stage 3 should already be extracted into this folder.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import sys, os, json, importlib.util
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Document Unwarping')
assert (ROOT / 'configs/datasets.yaml').is_file(), 'Stage 1 repository root was not found'
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])

Mounted at /content/drive
Repository: /content/drive/MyDrive/Document Unwarping
Python: 3.13.18


## 02. Confirm Stage 3 source installation

If you already extracted `document_unwarping_stage3_patch.zip` into the repository, skip uploading it. This cell checks the required files without modifying them.

In [2]:
REQUIRED_STAGE3 = [
    'configs/evaluation.yaml',
    'evaluation/metrics.py',
    'evaluation/runner.py',
    'evaluation/reference_metrics.py',
    'evaluation/protocols/uvdoc.py',
    'evaluation/protocols/docunet.py',
    'evaluation/protocols/dir300.py',
    'scripts/evaluate.py',
    'requirements-stage3.txt',
    'documentation/evaluation_protocols.md',
]
missing = [p for p in REQUIRED_STAGE3 if not (ROOT / p).is_file()]
assert not missing, 'Extract the Stage 3 patch into the repository first: ' + repr(missing)
print('Stage 3 source files found. Patch installation skipped.')

Stage 3 source files found. Patch installation skipped.


## 03. Check dependencies and device

Keep Colab's installed CUDA version of PyTorch. Only install missing non-Torch evaluation packages.

In [3]:
from importlib import util
required = {
    'cv2': 'opencv-python-headless>=4.8,<5',
    'yaml': 'PyYAML>=6,<7',
    'skimage': 'scikit-image>=0.22,<0.27',
    'hdf5storage': 'hdf5storage>=0.2,<0.3',
    'pytest': 'pytest>=8,<10',
}
missing_packages = [package for module, package in required.items() if util.find_spec(module) is None]
if missing_packages:
    import subprocess
    print('Installing missing non-Torch packages:', missing_packages)
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing_packages], check=True)
import torch, cv2, numpy as np, skimage, hdf5storage
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('NumPy:', np.__version__, 'OpenCV:', cv2.__version__, 'scikit-image:', skimage.__version__)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

Installing missing non-Torch packages: ['hdf5storage>=0.2,<0.3']
PyTorch: 2.14.1+cpu CUDA: None
NumPy: 2.4.6 OpenCV: 5.0.0 scikit-image: 0.25.2
Device: cpu CPU


## 04. Run all automated tests

These tests cover Stage 1 data contracts, Stage 2 inference and geometry, and Stage 3 evaluation. They do not require the UVDoc checkpoint or the full benchmark dataset.

In [4]:
import pytest
exit_code = pytest.main(['-q', 'tests'])
assert exit_code == 0, f'Automated tests failed with pytest exit code {exit_code}'
print('Stage 1/2/3 automated tests passed.')

........................................................................ [ 83%]
..............                                                           [100%]
86 passed in 33.74s
Stage 1/2/3 automated tests passed.


## 05. Inspect complete UVDoc Benchmark assets

The line metric requires `uvmap` MAT files, which were not part of the Stage 1 image smoke test. The visual metric also requires `warped_textures` and `texture_sample`. This cell reports each missing file instead of treating a missing archive member as a damaged manifest.

In [5]:
from data.manifests import load_records, manifest_path
from data.paths import PathResolver
from evaluation.protocols.uvdoc import inspect_uvdoc_assets

RESOLVER = PathResolver(ROOT)
UV_RECORDS = load_records('uvdoc_benchmark', ROOT)
assert len(UV_RECORDS) == 50, f'Expected 50 UVDoc Benchmark records, got {len(UV_RECORDS)}'
asset_rows = inspect_uvdoc_assets(UV_RECORDS, RESOLVER)
asset_fields = [k for k in asset_rows[0] if k.endswith('_available')]
for key in asset_fields:
    print(f'{key}: {sum(x[key] for x in asset_rows)}/50')
missing_asset_paths = sorted({row[k.replace('_available', '_logical_path')]
    for row in asset_rows for k in asset_fields if not row[k]})
print('Missing asset files:', len(missing_asset_paths))
for name in missing_asset_paths[:15]: print('  ', name)

metadata_available: 50/50
uvmap_available: 50/50
warped_texture_available: 50/50
reference_texture_available: 50/50
Missing asset files: 0


## 06. Optionally stage missing UVDoc Benchmark files

If required UVDoc Benchmark files are already present, this cell does nothing. If files are missing, it looks for an existing benchmark ZIP in `dataset/UVDoc/_archives/` and extracts **only those files**. It does not extract UVDoc Final training data or overwrite existing files.

If the archive is unavailable, the cell stops and tells you exactly which files must be obtained before running the line benchmark.

In [6]:
from zipfile import ZipFile
from shutil import copyfileobj

if missing_asset_paths:
    archive_dir = ROOT / 'dataset/UVDoc/_archives'
    archives = sorted(p for p in archive_dir.glob('*.zip') if 'benchmark' in p.name.lower())
    assert archives, 'Missing benchmark assets, and no UVDoc Benchmark ZIP exists in dataset/UVDoc/_archives'
    archive = archives[0]
    print('Using existing archive:', archive.name)
    staged = 0
    with ZipFile(archive) as zf:
        all_names = [n.replace(chr(92), '/') for n in zf.namelist()]
        for logical in missing_asset_paths:
            relative = logical.split('dataset/UVDoc/UVDoc_benchmark/', 1)[-1]
            candidates = [name for name in all_names if name.endswith('/UVDoc_benchmark/' + relative)
                          or name == 'UVDoc_benchmark/' + relative or name == relative]
            assert len(candidates) == 1, f'Could not uniquely locate {relative} in {archive.name}: {candidates}'
            target = ROOT / logical
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                temp = target.with_name(target.name + '.part')
                try:
                    with zf.open(candidates[0]) as source, temp.open('wb') as dest:
                        copyfileobj(source, dest)
                    temp.replace(target)
                finally:
                    temp.unlink(missing_ok=True)
                staged += 1
    print('Staged missing benchmark assets:', staged)
else:
    print('All required benchmark assets are already present.')
asset_rows = inspect_uvdoc_assets(UV_RECORDS, PathResolver(ROOT))
assert all(row[key] for row in asset_rows for key in asset_fields), 'Some benchmark files are still unavailable'

All required benchmark assets are already present.


## 07. Locate the verified official UVDoc checkout and Stage 2 sample run

The official model source was checked in Stage 2. The existing two-sample run contains `00010` and `00011` and will be used for the first metric parity test.

In [8]:
# 07. Restore the pinned official UVDoc source and locate Stage 2 predictions.

from pathlib import Path
import subprocess

from models.deep.uvdoc import load_uvdoc_config, verify_source, hash_file
from evaluation.runner import inspect_inference_run, load_evaluation_config

OFFICIAL_CFG = load_uvdoc_config()
COMMIT = OFFICIAL_CFG["source"]["commit"]
OFFICIAL_SOURCE = Path("/content/uvdoc_reference_stage2")

# The official checkout is stored in temporary Colab storage.
# Recreate it only when missing. Never clone inside the team repository.
if not OFFICIAL_SOURCE.exists():
    print("Official UVDoc checkout is missing. Restoring pinned source.")

    subprocess.run(
        [
            "git", "clone", "--quiet", "--depth", "1",
            OFFICIAL_CFG["source"]["repository"],
            str(OFFICIAL_SOURCE),
        ],
        check=True,
    )

    current_commit = subprocess.check_output(
        ["git", "-C", str(OFFICIAL_SOURCE), "rev-parse", "HEAD"],
        text=True,
    ).strip()

    if current_commit != COMMIT:
        subprocess.run(
            [
                "git", "-C", str(OFFICIAL_SOURCE),
                "fetch", "--quiet", "--depth", "1", "origin", COMMIT,
            ],
            check=True,
        )

        subprocess.run(
            [
                "git", "-C", str(OFFICIAL_SOURCE),
                "checkout", "--quiet", "--detach", COMMIT,
            ],
            check=True,
        )

# Verify the source and checkpoint, including their exact identities.
source_hashes = verify_source(OFFICIAL_SOURCE, COMMIT)

assert (OFFICIAL_SOURCE / "uvdocBenchmark_metric.py").is_file(), (
    "Official UVDoc line evaluator is missing."
)

CHECKPOINT = (
    OFFICIAL_SOURCE / OFFICIAL_CFG["checkpoint"]["relative_path"]
)

assert CHECKPOINT.is_file(), f"Checkpoint missing: {CHECKPOINT}"
assert CHECKPOINT.stat().st_size > 1024 * 1024, (
    "Checkpoint may be an unresolved Git LFS pointer."
)

actual_sha = hash_file(CHECKPOINT)

assert actual_sha == OFFICIAL_CFG["checkpoint"]["sha256"], (
    f"Checkpoint SHA256 mismatch: {actual_sha}"
)

print("Official UVDoc source: PASS")
print("Pinned commit:", COMMIT)
print("Checkpoint SHA256:", actual_sha)

# Locate the existing Stage 2 integration run.
STAGE2_PARENT = (
    ROOT
    / "runs/inference/stage2_integration_20261121T104333Z_cca74c"
)

TWO_SAMPLE_RUN = STAGE2_PARENT / "uvdoc_benchmark"

assert TWO_SAMPLE_RUN.is_dir(), (
    f"Stage 2 inference run missing: {TWO_SAMPLE_RUN}"
)

protocol = load_evaluation_config(ROOT)["protocols"][
    "uvdoc_lines_official_n25_v1"
]

bound, index, provenance = inspect_inference_run(
    ROOT, TWO_SAMPLE_RUN, protocol
)

sample_ids = {r["sample_id"] for r in bound}

assert sample_ids == {"00010", "00011"}, (
    f"Unexpected Stage 2 sample IDs: {sample_ids}"
)

print("Stage 2 inference artifacts: PASS")
print("Saved samples:", sorted(sample_ids))
print("Ready for the official line-metric parity test.")

Official UVDoc checkout is missing. Restoring pinned source.
Official UVDoc source: PASS
Pinned commit: 4c9b82b537057aff2526e6dd118a847cdd072e82
Checkpoint SHA256: 7e90861b8a516eb4bc51f84bd889cb77275743d2d1d3ca8091951ec9f2b7da23
Stage 2 inference artifacts: PASS
Saved samples: ['00010', '00011']
Ready for the official line-metric parity test.


## 08. Compare the new line metric with independent official code

Each line is calculated twice: once by Stage 3 and once by the pinned official `uvdocBenchmark_metric.py` in a separate process. This cell compares every line, including undefined values. It creates a new evaluation directory.

In [12]:
from evaluation.runner import evaluate_inference_run

def unique_run(prefix):
    return prefix + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]

TWO_LINE_RUN = ROOT / 'runs/evaluation' / unique_run('stage3_uvdoc_line_parity')
TWO_LINE_RESULT = evaluate_inference_run(
    ROOT, TWO_SAMPLE_RUN, 'uvdoc_lines_official_n25_v1', TWO_LINE_RUN,
    device=DEVICE, official_source=OFFICIAL_SOURCE,
    verify_parity_ids=['00010', '00011'],
)
print('Reference parity:', TWO_LINE_RESULT['parity'])
assert len(TWO_LINE_RESULT['parity']) == 2
assert all(x['status'] == 'PASS' for x in TWO_LINE_RESULT['parity']), 'Official line-metric parity did not pass'
print('Per-sample metrics:', TWO_LINE_RUN / 'metrics/per_sample.csv')

[1/2] UVDoc_Benchmark_rendered_00010: OK
[2/2] UVDoc_Benchmark_rendered_00011: OK
Reference parity: [{'sample_key': 'UVDoc_Benchmark_rendered_00010', 'status': 'PASS', 'reason': '', 'max_abs': 0.0, 'atol': 1e-05, 'lines': 50, 'reference_commit': '4c9b82b537057aff2526e6dd118a847cdd072e82'}, {'sample_key': 'UVDoc_Benchmark_rendered_00011', 'status': 'PASS', 'reason': '', 'max_abs': 0.0, 'atol': 1e-05, 'lines': 50, 'reference_commit': '4c9b82b537057aff2526e6dd118a847cdd072e82'}]
Per-sample metrics: /content/drive/MyDrive/Document Unwarping/runs/evaluation/stage3_uvdoc_line_parity_20261121T112121Z_8ed32762/metrics/per_sample.csv


In [10]:
import pandas as pd

display(
    pd.read_csv(TWO_LINE_RUN / "samples.csv")[
        ["sample_id", "evaluation_status", "reason"]
    ]
)

,sample_id,evaluation_status,reason
0,10,INVALID_INPUT,ValueError: Invalid UVDoc UV-map shape or valu...
1,11,INVALID_INPUT,ValueError: Invalid UVDoc UV-map shape or valu...


In [11]:
import importlib
import inspect

from evaluation.protocols import uvdoc

importlib.reload(uvdoc)

assert "NaN masks are permitted" in inspect.getsource(
    uvdoc.warp_synthetic_line
)

print("Stage 3 UV-map hotfix loaded.")

Stage 3 UV-map hotfix loaded.


## 09. Produce a separate full 50-image UVDoc inference run

Stage 2 only inferred two benchmark images. The evaluation itself never runs the neural network, so we generate the remaining predictions through the unchanged Stage 2 runner. This may take time and storage.

If you have already completed a 50-image Stage 2 run, set `EXISTING_UV50_RUN` to that directory. Otherwise leave it as `None`.

In [13]:
# Set this to an existing complete Stage 2 UVDoc Benchmark run to reuse it.
EXISTING_UV50_RUN = None

from scripts.run_inference import run_records
from models.deep.uvdoc import load_uvdoc_model

if EXISTING_UV50_RUN is not None:
    UV50_RUN = Path(EXISTING_UV50_RUN)
    print('Reusing existing inference run:', UV50_RUN)
else:
    UV50_RUN = ROOT / 'runs/inference' / unique_run('stage3_uvdoc50_inference')
    print('Creating full benchmark inference run:', UV50_RUN)
    model = load_uvdoc_model(OFFICIAL_CFG, device=DEVICE, source_dir=OFFICIAL_SOURCE)
    summary = run_records(UV_RECORDS, PathResolver(ROOT), model, UV50_RUN,
                          hash_file(manifest_path('uvdoc_benchmark', ROOT)),
                          export_benchmark_mat=True)
    print(summary)
    del model
assert UV50_RUN.is_dir()
bound50, reg50, meta50 = inspect_inference_run(ROOT, UV50_RUN, protocol)
assert len(bound50) == 50 and {r['sample_id'] for r in bound50} == {r.sample_id for r in UV_RECORDS}
print('Full run attempts:', len(bound50), 'OK:', sum(r['status']=='OK' for r in bound50))

Creating full benchmark inference run: /content/drive/MyDrive/Document Unwarping/runs/inference/stage3_uvdoc50_inference_20261121T112241Z_9a500faf
[1/50] UVDoc_Benchmark_rendered_00000: OK
[2/50] UVDoc_Benchmark_rendered_00001: OK
[3/50] UVDoc_Benchmark_rendered_00002: OK
[4/50] UVDoc_Benchmark_rendered_00003: OK
[5/50] UVDoc_Benchmark_rendered_00004: OK
[6/50] UVDoc_Benchmark_rendered_00005: OK
[7/50] UVDoc_Benchmark_rendered_00006: OK
[8/50] UVDoc_Benchmark_rendered_00007: OK
[9/50] UVDoc_Benchmark_rendered_00008: OK
[10/50] UVDoc_Benchmark_rendered_00009: OK
[11/50] UVDoc_Benchmark_rendered_00010: OK
[12/50] UVDoc_Benchmark_rendered_00011: OK
[13/50] UVDoc_Benchmark_rendered_00012: OK
[14/50] UVDoc_Benchmark_rendered_00013: OK
[15/50] UVDoc_Benchmark_rendered_00014: OK
[16/50] UVDoc_Benchmark_rendered_00015: OK
[17/50] UVDoc_Benchmark_rendered_00016: OK
[18/50] UVDoc_Benchmark_rendered_00017: OK
[19/50] UVDoc_Benchmark_rendered_00018: OK
[20/50] UVDoc_Benchmark_rendered_00019: OK
[2

## 10. Run the full UVDoc line benchmark

This computes 25 horizontal and 25 vertical line measurements for each of the 50 samples. It may run for several minutes. Results are saved after every sample, so already completed records remain inspectable if Colab disconnects.

The independent `00010` and `00011` parity results are recorded in this new run as well.

In [14]:
UV50_EVAL = ROOT / 'runs/evaluation' / unique_run('stage3_uvdoc50_lines')
UV50_RESULT = evaluate_inference_run(
    ROOT, UV50_RUN, 'uvdoc_lines_official_n25_v1', UV50_EVAL,
    device=DEVICE, official_source=OFFICIAL_SOURCE,
    verify_parity_ids=['00010', '00011'],
)
print('Status:', UV50_RESULT['status_counts'])
for row in UV50_RESULT['metrics']:
    print(row['metric_name'], 'valid', row['n_metric_ok'], '/', row['n_attempted'], 'mean', row['mean'])
assert UV50_RESULT['n_attempted'] == 50
assert len(UV50_RESULT['parity']) == 2 and all(r['status']=='PASS' for r in UV50_RESULT['parity'])
print('Full benchmark line report:', UV50_EVAL / 'report.md')

[1/50] UVDoc_Benchmark_rendered_00000: OK
[2/50] UVDoc_Benchmark_rendered_00001: OK
[3/50] UVDoc_Benchmark_rendered_00002: OK
[4/50] UVDoc_Benchmark_rendered_00003: OK
[5/50] UVDoc_Benchmark_rendered_00004: OK
[6/50] UVDoc_Benchmark_rendered_00005: OK
[7/50] UVDoc_Benchmark_rendered_00006: OK
[8/50] UVDoc_Benchmark_rendered_00007: OK
[9/50] UVDoc_Benchmark_rendered_00008: OK
[10/50] UVDoc_Benchmark_rendered_00009: OK
[11/50] UVDoc_Benchmark_rendered_00010: OK
[12/50] UVDoc_Benchmark_rendered_00011: OK
[13/50] UVDoc_Benchmark_rendered_00012: OK
[14/50] UVDoc_Benchmark_rendered_00013: OK
[15/50] UVDoc_Benchmark_rendered_00014: OK
[16/50] UVDoc_Benchmark_rendered_00015: OK
[17/50] UVDoc_Benchmark_rendered_00016: OK
[18/50] UVDoc_Benchmark_rendered_00017: OK
[19/50] UVDoc_Benchmark_rendered_00018: OK
[20/50] UVDoc_Benchmark_rendered_00019: OK
[21/50] UVDoc_Benchmark_rendered_00020: OK
[22/50] UVDoc_Benchmark_rendered_00021: OK
[23/50] UVDoc_Benchmark_rendered_00022: OK
[24/50] UVDoc_Benchm

## 11. Prepare and independently verify texture predictions

The official UVDoc visual benchmark uses unwarped textures, not the rectified photographic PNGs. This cell applies the saved coarse 2D grids to the benchmark's `warped_textures` images, then compares samples `00010` and `00011` with the original `utils.py` output pixel by pixel.

In [15]:
UV_TEXTURE_EVAL = ROOT / 'runs/evaluation' / unique_run('stage3_uvdoc50_texture')
UV_TEXTURE_RESULT = evaluate_inference_run(
    ROOT, UV50_RUN, 'uvdoc_texture_prepare_v1', UV_TEXTURE_EVAL,
    device=DEVICE, official_source=OFFICIAL_SOURCE,
    verify_parity_ids=['00010', '00011'],
)
print('Texture preparation:', UV_TEXTURE_RESULT['status_counts'])
print('Independent texture parity:', UV_TEXTURE_RESULT['parity'])
assert UV_TEXTURE_RESULT['n_attempted'] == 50
assert len(UV_TEXTURE_RESULT['parity']) == 2 and all(r['status']=='PASS' for r in UV_TEXTURE_RESULT['parity'])
print('Texture outputs:', UV_TEXTURE_EVAL / 'prepared_predictions/uwp_texture')

[1/50] UVDoc_Benchmark_rendered_00000: OK
[2/50] UVDoc_Benchmark_rendered_00001: OK
[3/50] UVDoc_Benchmark_rendered_00002: OK
[4/50] UVDoc_Benchmark_rendered_00003: OK
[5/50] UVDoc_Benchmark_rendered_00004: OK
[6/50] UVDoc_Benchmark_rendered_00005: OK
[7/50] UVDoc_Benchmark_rendered_00006: OK
[8/50] UVDoc_Benchmark_rendered_00007: OK
[9/50] UVDoc_Benchmark_rendered_00008: OK
[10/50] UVDoc_Benchmark_rendered_00009: OK
[11/50] UVDoc_Benchmark_rendered_00010: OK
[12/50] UVDoc_Benchmark_rendered_00011: OK
[13/50] UVDoc_Benchmark_rendered_00012: OK
[14/50] UVDoc_Benchmark_rendered_00013: OK
[15/50] UVDoc_Benchmark_rendered_00014: OK
[16/50] UVDoc_Benchmark_rendered_00015: OK
[17/50] UVDoc_Benchmark_rendered_00016: OK
[18/50] UVDoc_Benchmark_rendered_00017: OK
[19/50] UVDoc_Benchmark_rendered_00018: OK
[20/50] UVDoc_Benchmark_rendered_00019: OK
[21/50] UVDoc_Benchmark_rendered_00020: OK
[22/50] UVDoc_Benchmark_rendered_00021: OK
[23/50] UVDoc_Benchmark_rendered_00022: OK
[24/50] UVDoc_Benchm

## 12. Prepare DocUNet and DIR300 reference inputs

These diagnostic Stage 2 runs include only selected images. They can verify naming, pairing, missing-reference handling, and evaluation output structure. Published scores require full benchmark prediction sets and MATLAB/SIFT-flow reference code.

The documented document-64 orientation issue remains separate from the legacy DocUNet protocol.

In [16]:
diagnostic_cases = [
    ('docunet_crop', 'docunet_matlab_legacy_v1'),
    ('docunet_crop', 'docunet_matlab_corrected64_v1'),
    ('dir300', 'dir300_matlab_v1'),
]
reference_diagnostics = []
for subdir, protocol_name in diagnostic_cases:
    src = STAGE2_PARENT / subdir
    if not src.is_dir():
        print('Missing Stage 2 diagnostic run:', src)
        continue
    dest = ROOT / 'runs/evaluation' / unique_run('stage3_' + subdir + '_prep')
    result = evaluate_inference_run(ROOT, src, protocol_name, dest, device='cpu')
    reference_diagnostics.append((protocol_name, result['status_counts'], dest))
for protocol_name, status, path in reference_diagnostics:
    print(protocol_name, status, path)

[1/2] DocUNet_crop_38_1: OK
[2/2] DocUNet_crop_64_1: OK
[1/2] DocUNet_crop_38_1: OK
[2/2] DocUNet_crop_64_1: REVIEW_REQUIRED
[1/2] DIR300_dist_8: OK
[2/2] DIR300_dist_38: OK
docunet_matlab_legacy_v1 {'OK': 2} /content/drive/MyDrive/Document Unwarping/runs/evaluation/stage3_docunet_crop_prep_20261121T113354Z_ad907517
docunet_matlab_corrected64_v1 {'OK': 1, 'REVIEW_REQUIRED': 1} /content/drive/MyDrive/Document Unwarping/runs/evaluation/stage3_docunet_crop_prep_20261121T113359Z_87cc13d9
dir300_matlab_v1 {'OK': 2} /content/drive/MyDrive/Document Unwarping/runs/evaluation/stage3_dir300_prep_20261121T113359Z_f344f7e4


## 13. Check optional MATLAB dependencies

If MATLAB Engine, the reference scripts, and SIFT flow are unavailable, record that limitation. No Python approximation will be reported as the published MS-SSIM, LD, or AD score. A separate MATLAB-capable environment can later evaluate the full prepared outputs.

In [17]:
from evaluation.reference_metrics import matlab_availability
print('UVDoc MATLAB:', matlab_availability(OFFICIAL_SOURCE, 'uvdoc_visual_matlab_v1'))
print('DocUNet MATLAB:', matlab_availability(OFFICIAL_SOURCE, 'docunet_matlab_legacy_v1'))
print('DIR300 MATLAB uses the separate DocGeoNet reference checkout and SIFT-flow dependencies.')

UVDoc MATLAB: {'matlab_engine_installed': False, 'reference_script_present': True, 'siftflow_directory_present': True, 'protocol_id': 'uvdoc_visual_matlab_v1', 'ready_to_attempt': False}
DocUNet MATLAB: {'matlab_engine_installed': False, 'reference_script_present': True, 'siftflow_directory_present': True, 'protocol_id': 'docunet_matlab_legacy_v1', 'ready_to_attempt': False}
DIR300 MATLAB uses the separate DocGeoNet reference checkout and SIFT-flow dependencies.


## 14. Inspect reports and acceptance evidence

Check counts and failures before quoting any score. A mean based on missing or invalid samples must be accompanied by its coverage. Save these report paths for the team; do not commit generated run directories.

In [18]:
for label, path in [
    ('Two-sample parity', TWO_LINE_RUN),
    ('UVDoc 50-line evaluation', UV50_EVAL),
    ('UVDoc texture preparation', UV_TEXTURE_EVAL),
]:
    summary = json.loads((path / 'summary.json').read_text(encoding='utf-8'))
    print()
    print(label, '->', path)
    print('Attempted:', summary['n_attempted'], 'status:', summary['status_counts'])
    print('Parity:', summary['parity'])
    assert (path / 'artifacts.csv').is_file()
    assert (path / 'report.md').is_file()
print()
print('Stage 3 Python evaluation evidence is saved.')


Two-sample parity -> /content/drive/MyDrive/Document Unwarping/runs/evaluation/stage3_uvdoc_line_parity_20261121T112121Z_8ed32762
Attempted: 2 status: {'OK': 2}
Parity: [{'sample_key': 'UVDoc_Benchmark_rendered_00010', 'status': 'PASS', 'reason': '', 'max_abs': 0.0, 'atol': 1e-05, 'lines': 50, 'reference_commit': '4c9b82b537057aff2526e6dd118a847cdd072e82'}, {'sample_key': 'UVDoc_Benchmark_rendered_00011', 'status': 'PASS', 'reason': '', 'max_abs': 0.0, 'atol': 1e-05, 'lines': 50, 'reference_commit': '4c9b82b537057aff2526e6dd118a847cdd072e82'}]

UVDoc 50-line evaluation -> /content/drive/MyDrive/Document Unwarping/runs/evaluation/stage3_uvdoc50_lines_20261121T112450Z_d422fd05
Attempted: 50 status: {'OK': 50}
Parity: [{'sample_key': 'UVDoc_Benchmark_rendered_00010', 'status': 'PASS', 'reason': '', 'max_abs': 0.0, 'atol': 1e-05, 'lines': 50, 'reference_commit': '4c9b82b537057aff2526e6dd118a847cdd072e82'}, {'sample_key': 'UVDoc_Benchmark_rendered_00011', 'status': 'PASS', 'reason': '', 'm

## Handoff

Keep the three Stage 3 report directories and the full Stage 2 UVDoc inference run. Share `reference_validation/numerical_parity.csv`, `metrics/summary.csv`, and `report.md` when the team reviews results.

The Colab execution establishes the actual benchmark coverage and numerical compatibility. Source code and documentation remain in the repository. Large run outputs stay in `runs/` and should not be committed to Git.